# ☁️ Money Maker 911 — Colab GPU renderRenders the packed job (`job.zip`) on this GPU VM and zips the finished videos into `output/`.**Automatic mode:** run with `colab exec -s <session> -f mm911_render.ipynb` after uploading the job.**Manual mode:** upload `job.zip` yourself (Files panel), then *Runtime ▸ Run all*.

In [ ]:
!nvidia-smi || true

In [ ]:
# 1. unpack the job (or, in manual mode, pick the newest job.zip in /content)import glob, os, zipfilezips = sorted(glob.glob('/content/**/job.zip', recursive=True), key=os.path.getmtime)assert zips, 'job.zip not found - upload it first (Files panel or colab upload)'job_zip = zips[-1]print('using', job_zip)os.makedirs('/content/mm911', exist_ok=True)with zipfile.ZipFile(job_zip) as z:    z.extractall('/content/mm911')print('unpacked:', len(z.namelist()), 'files')

In [ ]:
# 2. install the renderer%cd /content/mm911/job!pip install -q pymunk pygame-ce numpy pillow imageio-ffmpeg!python -c "import pymunk, pygame, numpy; print('deps ok', pymunk.version, pygame.ver)"

In [ ]:
# 3. render (one video, or a whole batch when the job says so)import json, subprocess, sys, timejob = json.load(open('job.json'))count = int(job.get('count') or 0)cmd = ['python', 'main.py', 'batch', '--preset', 'preset.json', '--out', 'output'] if job.get('action') == 'batch' and count > 0 else ['python', 'main.py', 'render', '--preset', 'preset.json', '--out', 'output/video.mp4']if job.get('action') == 'batch' and count > 0:    cmd += ['--count', str(count)]print('$', ' '.join(cmd))t0 = time.time()r = subprocess.run(cmd)print(f'done in {time.time()-t0:.0f}s, exit {r.returncode}')assert r.returncode == 0, 'render failed - check the log above'

In [ ]:
# 4. pack the results for downloadimport shutilshutil.make_archive('/content/output', 'zip', '/content/mm911/job/output')print('ready: /content/output.zip')os.listdir('/content/mm911/job/output')